# Practical session 3: Selective sensing, parallel behaviors, environmental dynamics and multi-agent interactions

In the last practical session, we saw how to define, attach and detach a behavior on an agent. We implemented three distinct behaviors: `slow_down`, `fear` and `aggression`.

In this section we will see more sensing abilities the agent is equipped and will combine multiple behaviors together. We will also see how to make the agent consume resources that spawn in the environment. Finally, we will see how to attach those behaviors on multiple agents interacting together within a shared environment.

As usual, let's start by connecting this notebook to the simulation:

In [ ]:
from vivarium.controllers.vivarium_controller import VivariumController
controller = VivariumController.start_session(scene_name="session_3")

As seen in the scene map on the left, there are multiple entities in the current scene with different shapes and colors: a blue square as well as a number of orange and green circles. By default, square entities represent agents and circle entities represent objects. Agents and objects are two different types of entities: agents have motors and sensors while objects are mostly passive entities. 

In this scene we also have two subtypes of objects: large orange ones and smaller green ones. In this session, we call the orange objects *obstacles* and the green objects *resources*. 

## Selectively sensing other entities

To define a repertoire of interesting behaviors, we need the agent to selectively sense the proximity of different subtypes of entities. For example, we might want to define a behavior for obstacle avoidance and another one for collecting resources in the environment. The first behavior will require the proximeters to detect orange obstacles, whereas the second one will require the proximeters to detect the green resources. 

Since there is only one agent, the large blue square, let's create an alias variable to access it as in previous sessions:

In [ ]:
agent = controller.agents[0]

Each entity in the scene, either agents or objects, is associated with a *subtype*. The subtype of the green objects is `"resource"`, the subtype of the orange objects is `"obstacle"` and the subtype of the agent is `"agent"`, You can list the available subtypes in the current scene with:

In [ ]:
controller.subtypes

We can filter the result returned by the agent's proximeters by providing the argument `sensed_entities` to the `proximeters` function:

In [ ]:
left, right = agent.proximeters(sensed_entities=["obstacle"])
print(left, right)

Executing the cell above will return the proximeter activations only for the entities with the `"obstacle"` subtype, i.e. the orange circles in the scene. 

Note that the sensed entities can be occluded by other entities, whatever their subtype is. This means that if e.g. a `resource` object is closer than any `obstacle` object in the proximiter field of view, then the cell above will return 0 for that proximiter. This is somehow similar to how our own eyes sense objects: if you look at a tree but there is a wall between the tree and yourself, you won't see that tree. 

You can move an obstacle in the field of view of the proximiters using the drag and drop method (see [Session 1](https://github.com/flowersteam/vivarium/blob/upf2026/notebooks/sessions/session_1.ipynb), section "Connecting to the simulator", for how to drag and drop). Then re-execute the cell above to observe the change in the returned values. You can also check that the cell returns 0 when the closest object is a resource.

The `sensed_entities` argument requires a list of strings (`["obstacle"]` in the example above). In Python, a list is a collection of values separated by commas and surrounded by square brackets: `["obstacle"]` is therefore a list of only one element (the character string `"obstacle"`), whereas `["obstacle", "agent"]` is a list of two elements (the strings `"obstacle"` and `"agent"`). 


The agent's proximeters can detect multiple subtypes of entities. For instance, if we want the agent to sense either resources or obstacles we can write:

In [ ]:
left, right = agent.proximeters(sensed_entities=["resource", "obstacle"])
print(left, right)

In that case, each proximeter will sense the closest entity with one of the indicated subtypes (i.e. here the closest entity which either a resource or an obstacle). Try to drag and drop entities and re-executing the last cell above to understand how this behaves. Note that, in the current scene, calling `agent.proximeters(sensed_entities=["resource", "obstacle"])` is equivalent to calling `agent.proximeters()` since the only entities the agent can sense are anyway either resources or obstacles.

The `proximeter` function gives an error message if you provide a string that doesn't correspond to an existing subtype or that is spelled wrong, and ask you to select a type among the correct ones. For instance this will show an error:

In [ ]:
agent.proximeters(sensed_entities=["ressources"])  # typos in the entity subtype

The above cell returns an alarming error message, it's normal because we intentionally made a typo in the subtype (writing `"ressources"` instead of `"resource"`). In the last line of the error, the valid subtypes are indicated. Thus, we can correct it:

In [ ]:
agent.proximeters(sensed_entities=["resource"])  # Now we use the correct spelling

Using this mechanism for selectively sensing objects, we can now define behaviors targeting specific object subtypes. For experiencing this let's first define a few behaviors as explained below.

In this [slide from the class](https://docs.google.com/presentation/d/1s6ibk_ACiJb9CERJ_8L_b4KFu9d04ZG_htUbb_YSYT4/edit#slide=id.g31e1b425a3_0_0), we saw how different ways of connecting proximeter activations to motor activations results in different types of behaviors. Two of these behaviors result in moving closer to a sensed object (`love` and `aggression`) while the two others result in moving away from a sensed object (`shyness` and `fear`). Among these four behaviors, two of them use excitatory connections, resulting in the agent moving faster when it is closer to the sensed object (as in `aggression` and `fear`) ; while the two others use inhibitory connections, resulting in the agent moving slower when it is closer to an object (as in `love` and `shyness`).

Let's for instance define a behavior for avoiding obstacles. Which of the four behaviors illustrated in the slide is best suited for this? We want the agent to move away from the obstacles, so `shyness` and `fear` are our two candidates. We also want the agent to avoid collisions with the obstacles, so moving slower when the obstacle is closer makes sense here. We have our winner behavior: `shyness` is the best candidate for a behavior that aims at avoiding obstacles. 

As illustrated on the slide, the `shyness` behavior consists in crossed inhibitory connections. By "crossed" we mean that each proximeter, left or right, is connected to the wheel on the **opposite** side of the agent (left-to-right, right-to-left). By "inhibitory", we mean that the more a proximeter is activated, the **less** the wheel it is connected to will be activated. Since both the proximeter and motor activations are values bounded between 0 and 1, the `shyness` behavior therefore corresponds to:

$$m_L = 1 - s_R$$
$$m_R = 1 - s_L$$

where $s_L$ (resp. $s_R$) corresponds to the left (resp. right) proximeter sensor activation ; and $m_L$ (resp. $m_R$) corresponds to the left (resp. right) wheel motor activation. This way, the more a given proximeter is activated, the less the opposite wheel will be activated.

Using the way to define behaviors we have seen in [Session 2](https://github.com/flowersteam/vivarium/blob/upf2026/notebooks/sessions/session_2.ipynb), the `shyness` behavior can therefore be written as:

In [ ]:
# Shyness behavior

def shyness(agent):
    # First we read the sensor values
    left_sensor, right_sensor = agent.proximeters()

    # Then we compute the corresponding motor activations from the above formula
    left_motor = 1 - right_sensor
    right_motor = 1 - left_sensor

    # Finally we return the left and right motor activations, in this order
    return left_motor, right_motor
    

Now we can attach the `shyness` behavior to the agent:

In [ ]:
agent.attach_behavior(shyness)

Now the agent should navigate in the scene map, making its best to avoid any entity it encounters. Given the above definition of the `shyness` behavior, the agent moves at full speed when no object is in its field of view, reduces its speed if it approaches an object and turns away from the sensed object. It might still enter in contact with objects if it is surrounded by many of them, moving at a very slow speed in this case.

However, this behavior senses all objects in the scene -- i.e. both green resources and orange obstacles. If we instead want to only avoid obstacles and not resources, we can use the selective sensing mechanism we have seen above. Remember that for sensing only obstacles in the scene we write:

In [ ]:
agent.proximeters(sensed_entities=["obstacle"])

Therefore we can define an obstacle avoidance behavior as:

In [ ]:
# The obstacle avoidance behavior is similar to the shyness behavior
# except that it only senses the obstacles, not the resources

def obstacle_avoidance(agent):
    # First we read the sensor values, sensing only the entities with subtype "obstacle"
    left_sensor, right_sensor = agent.proximeters(sensed_entities=["obstacle"])

    # Then we compute the corresponding motor activations from the above formula
    left_motor = 1 - right_sensor
    right_motor = 1 - left_sensor

    # Finally we return the left and right motor activations, in this order
    return left_motor, right_motor
    

Now we can detach the `shyness` behavior which is currently being executed and attach the new `obstacle_avoidance` behavior we have just defined:

In [ ]:
# Detach all the behaviors currently attached to the agent
agent.detach_all_behaviors(stop_motors=True)

# Attach the obstacle_avoidance behavior we have just defined
agent.attach_behavior(obstacle_avoidance)

The agent should now smoothly navigate between the obstacles in the scene and don't care about resources (potentially colliding with resources). 

## Executing multiple behaviors in parallel

As the `obstacle_avoidance` behavior only senses the orange obstacles, the agent currently does not react to green resources. Let's implement a behavior that makes the agent forage for resources and execute it together with the obstacle avoidance behavior.

**Q1:** Define a behavior allowing the agent to forage for resources, let's call it `foraging`. The agent has to orient itself toward resources only, with a speed proportional to the proximiter activations (the closer the resource, the higher the speed) 

- *Tip 1:* First think about which of the [four behaviors we have seen in class](https://docs.google.com/presentation/d/1s6ibk_ACiJb9CERJ_8L_b4KFu9d04ZG_htUbb_YSYT4/edit?usp=sharing) is best suited for foraging. 
- *Tip 2:* You already saw how to detect only obstacles in the obstacle avoidance behavior. Here the agent will instead have to detect only resources. The subtype of the resources is `"resource"`.

In [ ]:
def foraging(agent):
    # your code here
    

Agents can actually run several behaviors in parallel. Currently, only the `obstacle_avoidance` behavior should be attached to the agent. We can verify it with:

In [ ]:
agent.print_behaviors()

Let's also attach the `foraging` behavior you have just define:

In [ ]:
agent.attach_behavior(foraging)

Note that this time we haven't detached the previous behavior. In consequence, both the `obstacle_avoidance` behavior we previously attached and the `foraging` behavior are now executed together on the agent. You should therefore see the agent avoiding obstacles while being attracted by the resources. We can check that the two behaviors are indeed attached with:

In [ ]:
agent.print_behaviors()

When multiple behaviors are executed in parallel on the same agent, the motor activations of the wheels correspond to the average of the motor activations returned by each behavior. This averaging is implemented internally, you don't need to worry about it when you define the behaviors.

**Q2:** Considering that both the `obstacle_avoidance` and the `foraging` behaviors are currently executed in parallel on the agent, what should be the values of the left and right wheel activations in the situations expressed below? 

*Tip 1:* To answer you need to first think about the motor activations returned by each behavior independently, then average them. Just do it in your mind, no need to code here. 

*Tip 2:* Remember that the proximeter activations are bounded between 0 and 1. It is 0 when there is no entity of the corresponding subtype in the field of view ; and 1 when it is fully activated.

What are the left and right motor activations when: 

- There is no object in the agent's field of view:

*your answer here*

- There is a green resource maximally activating the left proximeter and no object sensed by the right proximeter:

*your answer here*

- There is no object sensed by the left proximeter and an orange obstacle maximally activating the right proximeter:

*your answer here*

- There is an orange obstacle maximally activating the left proximeter and a green resource maximally activating the right proximeter:

*your answer here*

## Environmental dynamics

For now, the scene in which the agent is behaving is quite static: the agent interacts with the existing objects, but there is nothing that appears or disappears in the environment. We are now going to see how we can generate resources appearing at random positions in the environment and disappearing whenever an agent consumes them. 

### Consumption mechanism

The consumption mechanism specifies how some entities can consume other entities, making them disappear from the environment. For this we need to define:

- A source subtype, specifying what subtype is consuming something. As we want agents to consume resources, the source subtype will be `"agent"`.
- A target subtype, specifying which subtype is being consumed by the source subtype. As we want agents to consume resources, the target subtype will be `"resource"`.
- A consumption range, specifying what is the distance between two entities at which the consumption is triggered.

Programmatically we do it this way:

In [ ]:
# We first specify the "source subtype" of the consumption mechanism, 
# i.e. what subtype will be consuming other entities:
controller.consumption.source_subtype = "agent"

# Then we specify the "target subtype" of the consumption mechanism,
# i.e. what subtype is being consumed by the source subtype:
controller.consumption.target_subtype = "resource"

# Then we specify the distance range at which the consumption is triggered
controller.consumption.range = 1

# Finally we activate the consumption mechanism with:
controller.consumption.start = True

Now you should see the agent consuming resources, meaning that the resources are disappearing whenever the agent is close enough to them (at a distance lesser than 1 unit). After some time, all resources will have been consumed by the agent.

### Spawning mechanism

The spawning mechanism enables to regularly spawn new entities in the environment. We are going to use it for regularly spawning new resources in order to avoid their depletion. For this we need to define:
- The subtype of entities that will be spawned. Here we want to spawn resources, so the subtype will be `"resource"`.
- The time interval at which resources will spawn. Below we spawn them every 300 time steps. Higher values will make resources spawn less frequently (i.e. more time between each resource spawn).

In [ ]:
# We first specify what subtype will be spawning
controller.spawn.subtype = "resource"

# Then we specify the time interval at which resources will spawn
# Higher values will make resources spawn less frequently
# (i.e. more time between each resource spawn)
controller.spawn.period = 300

# Finally we activate the spawning mechanism
controller.spawn.start = True

Now you should see resources spawning at random positions in the environment at regular intervals, while the agent is consuming them. The resources will spawn until the maximum number of resources is reached in the environment (the maximum is 15 resources in the current scene). If this maximum number is reached and the agent consumes a resource, another one will appear at a random position in the scene.

#### Controlling spawning positions

You can also control the position range where the resources will appear with the `controller.spawn.position_range` parameter. This parameter accepts a list of 4 values: `(x_min, x_max, y_min, y_max)` where `x_min` and `x_max` are the minimum and maximum `x` coordinates of the spawning area, and `y_min` and `y_max` are the minimum and maximum `y` coordinates of the spawning area. Note that the current scene has a size of 100 by 100, as shown by the numbers on the x and y axes of the map. 

For example, to make resources appear only in the top-left quarter of the map (i.e. the area between x=0 and x=50, and y=50 and y=100) we write:

In [ ]:
# The position_range parameter takes a list of four values (x_min, x_max, y_min, y_max)
# Here it correspond to the top-left quarter of the map
controller.spawn.position_range = (0, 50, 50, 100)

Now the resources will spawn only in the top-left corner of the map. Note that the resources that were already in the map might still be outside of this area until the agent consumes them.

**Q3:** Make resources appear in the bottom-right quarter of the map:

In [ ]:
# Your code here


To test your answer to the last question you might want to first remove all currently existing resources from the environment. You can achieve this with:

In [ ]:
for obj in controller.objects:  # Iterate through all objects
    if obj.subtype == "resource":  # If the object is a resource
        obj.exists = False  # Mark this object as non-existing (i.e. it won't show in the scene map)

Side note: If needed you can reuse the logic of the cell above to change the attribute of entities in batch. For instance if we want to change the color of all obstacles to purple we can write:

In [ ]:
for obj in controller.objects:  # Iterate through all objects
    if obj.subtype == "obstacle":  # If the object is an obstacle
        obj.color = "purple"  # Set the color of the object to purple

### Starting and stopping the consumption and spawning mechanims on the fly

We can stop the consumption mechanism with:

In [ ]:
controller.consumption.start = False

Now agent are no longer consuming resources, they just collide with them.

If we want to start it again we execute:

In [ ]:
controller.consumption.start = True

Similarly for the spawning mechansim we can stop it with:

In [ ]:
controller.spawn.start = False

Now resources are no longer spawning in the environment. To start it again:

In [ ]:
controller.spawn.start = True

## Dealing with multiple agents

This section explains how to deal with multiple agents and how to attach different behaviors to them.

At the moment there is only one existing agent that we can see in the scene. But there is actually another one that is not "existing" yet. We can see it with:

In [ ]:
controller.agents

The cell above returns a list with two elements in it, each one corresponding to an agent (internally each agent corresponds to a Python object of type `AgentController`). The first element in the list is accessed with `controller.agent[0]` (in Python list indices starts at 0) and corresponds to the agent we have manipulated above (using the alias variable `agent` which was set to `controller.agent[0]` at the start of this notebook). We can check that this first agent indeed exists with:

In [ ]:
controller.agents[0].exists

which is `True`, meaning that this agent does exist (it is the one we see in the interface). Let's check it for the second agent (index `1` of the list):

In [ ]:
controller.agents[1].exists

which is `False`, meaning that this agent does not exist yet. Non-existing entities are not visible in the scene map and they do not interact with other entities.

We can make this agent exist with:

In [ ]:
# Make the agent with index 1 exist
controller.agents[1].exists = True

Now you should see a second agent in the interface. 

Let's rename our original `agent` to `agent_0` and create an alias `agent_1` for the second one to easily access them:

In [ ]:
agent_0 = controller.agents[0]
agent_1 = controller.agents[1]

`agent_0` is the same as the one we called `agent` before.

Let's change the color of the second agent to be red, so that we can more easily distinguish it in the scene map:

In [ ]:
agent_1.color = 'red'

Now you have access to the two agents through the variables `agent_0` and `agent_1` (these variables names are arbitrary, you can choose whatever you want, e.g. `predator` and `prey`). 

We can attach and start behaviors on each agent independently, in the same way as we did before, simply using either the `agent_0` and `agent_1` variables instead of only the `agent` one as before. As an example, let's say we want to attach the `obstacle_avoidance` behavior we have defined above to `agent_0`, and both the `obstacle_avoidance` and the `foraging` behaviors to `agent_1`. 

In [ ]:
# detach the behaviors currently executed on agent_0, if any
agent_0.detach_all_behaviors()
# only attach the obstacle_avoidance behavior to agent_0
agent_0.attach_behavior(obstacle_avoidance)

# detach the behaviors currently executed on agent_1, if any
agent_1.detach_all_behaviors()
# attach the obstacle_avoidance and foraging behaviors to agent_1
agent_1.attach_behavior(obstacle_avoidance)
agent_1.attach_behavior(foraging)

Now `agent_0` (the blue one) should avoid obstacles and don't care about resources, while `agent_1` (the red one) avoids obstacles as well and also forage for resources.

We can check it is indeed the case by printing the behaviors currently executed on `agent_0`:

In [ ]:
# Print the behaviors attached to agent_0
print("agent_0 behaviors:")
agent_0.print_behaviors()

And the behaviors currently executed on `agent_1`:
:

In [ ]:
# Print the behaviors attached to agent_1
print("agent_1 behaviors:")
agent_1.print_behaviors()

Let's detach the behaviors and stop the motors of both agents with the following cell. Using this `for` loop on the `controller.agents` list and the `detach_behaviors` function enables to detach the behaviors of all the agents at once.

In [ ]:
# detach all the behaviors and stop the motors of all agents:
for agent in controller.agents: # For all agents
    # Detach all their behaviors and stop their motors
    agent.detach_all_behaviors(stop_motors=True)

### Simulating prey-predator interactions

Using all the mechanisms we have seen in this session, let's now implement a simple prey-predator simulation where:

- One of the two agents is called `prey` and the other agent is called `predator`.
- The `prey` forages for the green resources and consume them.
- Resources regularly spawn in the environment.
- The `predator` agent executes the `aggression` behavior toward the `prey`.
- The `prey` agent executes the `fear` behavior toward the `predator`.
- Both agents avoid obstacles.

Let's first rename our two agents as `prey` and `predator`:

In [ ]:
prey = controller.agents[0]
predator = controller.agents[1]

Let's also change their physical attributes to distinguish them more easily. The prey will be smaller than the predator (`diameter` attribute) but will move faster (`max_speed` attribute):

In [ ]:
prey.diameter = 4  # Decrease the diameter of the prey
prey.max_speed = 2.0  # Increase its maximum speed
prey.color = 'cyan'  # Change its color to cyan

predator.diameter = 6  # The predator is larger than the prey
predator.max_speed = 1.0  # But it moves slower
predator.color = 'red'  # Change its color to red

**Q4:** Define the `fear` and `aggression` behaviors towards other agents in the cell below.

- *Tip 1:* You can can find the illustation of each behavior in [the slide]('https://docs.google.com/presentation/d/1s6ibk_ACiJb9CERJ_8L_b4KFu9d04ZG_htUbb_YSYT4/edit?usp=sharing). 
- *Tip 2:* You want to target these behaviors toward other agents. The agents are of subtype `"agent"`. An agent cannot sense itself, only the other agents.

In [ ]:
def fear(agent):
    # your code here
    

def aggression(agent):
    # your code here
    

**Q5:** Detach all behaviors on all agents. Then attach the `obstacle_avoidance`, `foraging` and `fear` behaviors to the `prey` agent. Finally, attach the `obstacle_avoidance` and the `aggression` behavior to the `predator` agent. The spawning and consumption mechanisms also need to be started (normally they already are, but if not just can just use the commands we saw earlier in this section).

In [ ]:
# attach the aggression and obstacle_avoidance behaviors to agent_0

# your code here


# attach the fear and obstacle_avoidance behaviors to agent_1

# your code here


You should now observe a simple prey-predator interaction, where the `predator` agent tries to catch the `prey` agent while the `prey` agent tries to escape from the `predator` agent and actively forage for resources.
They should be both avoiding obstacles as well.

## Modifying the proximeter fields of view

Because the proximeters of `prey` are only directed in the forward direction, it is really hard to avoid the `predator` when it comes from behind it (it cannot see it in that case). Additionally, the `predator` has a pretty bad vision because it can't see the `prey` from very far.

We can improve this by changing the distance ranges and the angles of the fields of view using these agent attributes:

- The `proxs_dist_max` attribute specifies the maximum distance at which the proximeters can detect entities.
- The `proxs_cos_min` attribute is the cosine of the maximum angle at which the proximeter can detect other entities. It is a value is between -1 and 1. The closer this value is to 1, the narrower the proximeter field of view, and inversely for -1. By default this value is 0, corresponding to a field of view covering a semi-disk (this is currently the case, as visualized in light red in the interface).

Let's modify the field of view of the agents:

In [ ]:
# decrease the distance range and increase the angle of the prey proximeters
prey.proxs_dist_max = 10  # The prey agent can only sense at a distance of 10 units
prey.proxs_cos_min = -0.99 # But it has a very wide vision (almost 360°)

In [ ]:
# increase the distance range and decrease the angle of the predator proximeters
predator.proxs_dist_max = 20. # The proximeters of the predator agent can sense twice as far as the prey agent, up to a distance of 20 units
predator.proxs_cos_min = 0.9  # But its vision is very narrow

You can observe on the scene map that the fields of view of each agent has changed. The predator-prey interaction should now be a bit more lively. 

That's it for today. You can now close the app (`Ctrl-C` in the terminal window that spawns when you opened it). Don't forget to save this notebook before. Once ready you can deliver it on Aula Global.